# ScholarshipRegion.com — Scraping Pipeline

This notebook walks through scraping scholarships from ScholarshipRegion.com and shaping the data to match the **Mibegnon** `Scholarship` database schema.

## Approaches covered
1. **WordPress REST API** — structured JSON, no HTML parsing (preferred)
2. **BeautifulSoup fallback** — for fields only available in article HTML
3. **Data cleaning & schema mapping** — align scraped data to Prisma model
4. **Export** — to JSON / CSV for seeding the database

## 0. Install dependencies

In [ ]:
%pip install requests beautifulsoup4 pandas tqdm

In [ ]:
import requests
import json
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime

BASE_URL = "https://www.scholarshipregion.com"
API_BASE = f"{BASE_URL}/wp-json/wp/v2"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; MibegnonBot/1.0)"
}

print("Ready.")

---
## 1. WordPress REST API — Discover Categories

ScholarshipRegion is WordPress-powered, so the REST API is available out of the box.  
First, let's find the category IDs we care about.

In [ ]:
def get_categories(search: str = "") -> list:
    params = {"per_page": 100, "search": search}
    r = requests.get(f"{API_BASE}/categories", headers=HEADERS, params=params)
    r.raise_for_status()
    return r.json()

# List all top-level scholarship categories
cats = get_categories(search="scholarships")
for c in cats:
    print(f"[{c['id']:>5}]  {c['name']:40s}  ({c['count']} posts)  slug: {c['slug']}")

## 2. Fetch All Posts for a Category

WordPress paginates at 100 posts max per page. We loop until we have everything.

In [ ]:
def fetch_posts_by_category(category_id: int, max_pages: int = 20) -> list:
    """
    Fetch all posts for a given WP category ID.
    Returns a list of raw WP post objects.
    """
    posts = []
    for page in range(1, max_pages + 1):
        params = {
            "categories": category_id,
            "per_page": 100,
            "page": page,
            "_fields": "id,date,modified,slug,link,title,excerpt,content",
        }
        r = requests.get(f"{API_BASE}/posts", headers=HEADERS, params=params)
        if r.status_code == 400:  # WP returns 400 when page is out of range
            break
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        posts.extend(batch)
        print(f"  Page {page}: {len(batch)} posts (total so far: {len(posts)})")
        time.sleep(0.5)  # be polite
    return posts


# Set CATEGORY_ID after running Cell 1 above
CATEGORY_ID = None  # e.g. CATEGORY_ID = 42

if CATEGORY_ID:
    raw_posts = fetch_posts_by_category(CATEGORY_ID)
    print(f"
Total posts fetched: {len(raw_posts)}")
else:
    print("Set CATEGORY_ID first (run Cell 1 to find category IDs).")

## 3. Fetch All Posts (No Category Filter)

If you want every scholarship post on the site in one go:

In [ ]:
def fetch_all_posts(max_pages: int = 50) -> list:
    posts = []
    for page in tqdm(range(1, max_pages + 1), desc="Fetching pages"):
        params = {
            "per_page": 100,
            "page": page,
            "_fields": "id,date,modified,slug,link,title,excerpt,content",
        }
        r = requests.get(f"{API_BASE}/posts", headers=HEADERS, params=params)
        if r.status_code == 400:
            break
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        posts.extend(batch)
        time.sleep(0.3)
    return posts

# Uncomment to run:
# raw_posts = fetch_all_posts()
# print(f"Total: {len(raw_posts)} posts")

---
## 4. BeautifulSoup — Parse Article HTML

The WP API returns `content.rendered` (raw HTML). We use BeautifulSoup to pull out
**deadline**, **amount**, **country**, and **image** from each article body.

In [ ]:
def strip_html(html: str) -> str:
    """Remove all HTML tags and return plain text."""
    return BeautifulSoup(html, "html.parser").get_text(separator=" ", strip=True)


def extract_deadline(text: str):
    """Extract a deadline date string from article text."""
    patterns = [
        r"[Dd]eadline[:\s]+([A-Z][a-z]+ \d{1,2},?\s*\d{4})",
        r"[Dd]eadline[:\s]+(\d{1,2} [A-Z][a-z]+ \d{4})",
        r"[Cc]losing [Dd]ate[:\s]+([A-Z][a-z]+ \d{1,2},?\s*\d{4})",
        r"(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})",
    ]
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            return m.group(1).strip()
    return None


def extract_amount(text: str):
    """Extract funding amount or type (fully funded, ,000, etc.)."""
    m = re.search(r"(fully funded|partial(?:ly)? funded)", text, re.IGNORECASE)
    if m:
        return m.group(1).capitalize()
    m = re.search(r"[\$€£]\s?[\d,]+(?:\.\d{2})?(?:\s?(?:USD|EUR|GBP))?", text)
    if m:
        return m.group(0).strip()
    return None


def extract_country(text: str):
    """Naive country extraction from article text."""
    m = re.search(r"[Ss]tudy in ([A-Z][a-zA-Z\s]+?)(?:[,.]|\s(?:for|and|with|to))", text)
    if m:
        return m.group(1).strip()
    m = re.search(r"[Hh]osted (?:in|by) ([A-Z][a-zA-Z\s]+?)(?:[,.])", text)
    if m:
        return m.group(1).strip()
    return None


print("Extraction helpers defined.")

## 5. Parse a Single Post → Mibegnon Schema

The Mibegnon `Scholarship` Prisma model has these fields:
```
id, title, description, amount, deadline,
country, academicLevels, fields,
sourceUrl, imageUrl, createdAt, updatedAt
```
We map each WP post to this shape.

In [ ]:
def parse_post(post: dict) -> dict:
    """Map a raw WordPress post to the Mibegnon Scholarship schema."""
    title        = strip_html(post["title"]["rendered"])
    excerpt      = strip_html(post["excerpt"]["rendered"])
    content_html = post["content"]["rendered"]
    content_text = strip_html(content_html)

    # Grab first <img> as imageUrl
    soup    = BeautifulSoup(content_html, "html.parser")
    img_tag = soup.find("img")
    image_url = img_tag["src"] if img_tag and img_tag.get("src") else None

    return {
        "title":          title,
        "description":    excerpt or content_text[:400],
        "amount":         extract_amount(content_text),
        "deadline":       extract_deadline(content_text),  # raw string, normalised later
        "country":        extract_country(content_text),
        "academicLevels": [],   # filled in Cell 7
        "fields":         [],   # filled in Cell 7
        "sourceUrl":      post["link"],
        "imageUrl":       image_url,
        "_wp_id":         post["id"],
        "_wp_date":       post["date"],
    }


# Quick test (assumes raw_posts is populated):
# sample = parse_post(raw_posts[0])
# print(json.dumps(sample, indent=2, ensure_ascii=False))

## 6. Infer Academic Levels & Fields of Study

The Prisma `EducationLevel` enum:  
`TROISIEME | SECONDE | PREMIERE | TERMINALE | LICENCE | MASTER | DOCTORAT | PROFESSIONNEL`

In [ ]:
LEVEL_KEYWORDS = {
    "LICENCE":       ["bachelor", "undergraduate", "licence", "bsc", "b.sc"],
    "MASTER":        ["master", "msc", "m.sc", "mba", "postgraduate", "graduate"],
    "DOCTORAT":      ["phd", "doctorate", "doctoral", "d.phil"],
    "PROFESSIONNEL": ["professional", "vocational", "training", "fellowship"],
    "TERMINALE":     ["high school", "secondary", "terminale"],
}

FIELD_KEYWORDS = {
    "Sciences":          ["science", "stem", "biology", "chemistry", "physics", "math"],
    "Ingenierie":        ["engineering", "computer science", "technology"],
    "Medecine":          ["medicine", "medical", "health", "nursing", "pharmacy"],
    "Droit":             ["law", "legal", "jurisprudence"],
    "Commerce":          ["business", "economics", "finance", "management", "mba"],
    "Arts":              ["art", "design", "music", "architecture", "creative"],
    "Sciences Sociales": ["social", "sociology", "psychology", "political"],
    "Environnement":     ["environment", "climate", "sustainability", "agriculture"],
}

def infer_levels(text: str) -> list:
    t = text.lower()
    found = [lvl for lvl, kws in LEVEL_KEYWORDS.items() if any(kw in t for kw in kws)]
    return found if found else ["LICENCE"]  # sensible default

def infer_fields(text: str) -> list:
    t = text.lower()
    return [f for f, kws in FIELD_KEYWORDS.items() if any(kw in t for kw in kws)]


# Quick test
test = "Gates Cambridge Scholarship 2026 for Master and PhD Students in Science"
print("Levels:", infer_levels(test))
print("Fields:", infer_fields(test))

## 7. Full Parse Pipeline

In [ ]:
def parse_all(raw_posts: list) -> list:
    results = []
    for post in tqdm(raw_posts, desc="Parsing posts"):
        try:
            item = parse_post(post)
            combined = item["title"] + " " + (item["description"] or "")
            item["academicLevels"] = infer_levels(combined)
            item["fields"]         = infer_fields(combined)
            results.append(item)
        except Exception as e:
            print(f"  Skipped post {post.get('id')}: {e}")
    return results


# Run when raw_posts is ready:
# scholarships = parse_all(raw_posts)
# print(f"Parsed {len(scholarships)} scholarships")

---
## 8. Inspect Results as a DataFrame

In [ ]:
# df = pd.DataFrame(scholarships)
# df[["title", "country", "amount", "deadline", "academicLevels"]].head(20)

In [ ]:
# Coverage check — how many posts had each field extracted?
# for col in ["country", "amount", "deadline", "imageUrl"]:
#     filled = df[col].notna().sum()
#     print(f"{col:15s}: {filled}/{len(df)} ({filled/len(df)*100:.0f}%)")

---
## 9. Export to JSON

In [ ]:
def export_json(scholarships: list, path: str = "scholarships_seed.json"):
    # Strip internal _wp_* keys before export
    clean = [{k: v for k, v in s.items() if not k.startswith("_")} for s in scholarships]
    with open(path, "w", encoding="utf-8") as f:
        json.dump(clean, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(clean)} scholarships to {path}")

# export_json(scholarships)

---
## 10. Prisma Seed Script (Next Step)

Once you have `scholarships_seed.json`, load it into the Mibegnon DB:

```ts
// prisma/seed.ts
import { PrismaClient } from "@prisma/client";
import data from "../scholarships_seed.json";

const prisma = new PrismaClient();

async function main() {
  for (const s of data) {
    await prisma.scholarship.upsert({
      where:  { sourceUrl: s.sourceUrl },
      update: s,
      create: s,
    });
  }
  console.log(`Seeded ${data.length} scholarships`);
}

main().finally(() => prisma.$disconnect());
```

```bash
npx tsx prisma/seed.ts
```

Add to `package.json`:
```json
"prisma": { "seed": "tsx prisma/seed.ts" }
```

---
## Summary

| Cell | What it does |
|------|--------------|
| 1 | List WP categories → find category IDs |
| 2 | Fetch posts by category (paginated) |
| 3 | Fetch all posts site-wide (no filter) |
| 4 | BS4 helpers: extract deadline, amount, country, image |
| 5 | Map WP post → Mibegnon `Scholarship` object |
| 6 | Infer `academicLevels` and `fields` from text |
| 7 | Run full pipeline on all raw posts |
| 8 | Inspect results with Pandas |
| 9 | Export to `scholarships_seed.json` |
| 10 | Prisma seed script to push into the DB |